In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")

# Load email dataset
df_email = pd.read_csv('../dataset/email/phishing_email.csv')

print("=" * 50)
print(f"Dataset Shape: {df_email.shape[0]} rows, {df_email.shape[1]} columns")
print("=" * 50)
print("Column Names:", df_email.columns.tolist())

# Inspect first 3 rows
df_email.head(3)

Dataset Shape: 39154 rows, 7 columns
Column Names: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']


,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1


In [3]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# 0. Automatically create output directory
os.makedirs('../models/email', exist_ok=True)

# 1. Clean missing values and combine 'subject' and 'body'
df_email['subject'] = df_email['subject'].fillna('')
df_email['body'] = df_email['body'].fillna('')
df_email['full_text'] = df_email['subject'] + " " + df_email['body']

# Check target distribution
counts = df_email['label'].value_counts()
print(f"Legitimate Emails (0): {counts.get(0, 0)} | Phishing Emails (1): {counts.get(1, 0)}")

# 2. Train-Test Split (80% Train, 20% Test)
X = df_email['full_text']
y = df_email['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. TF-IDF Feature Extraction
print("\nExtracting features using TF-IDF...")
tfidf_email = TfidfVectorizer(
    max_features=3000, 
    stop_words='english', 
    ngram_range=(1, 1), 
    sublinear_tf=True
)

X_train_tfidf = tfidf_email.fit_transform(X_train)
X_test_tfidf = tfidf_email.transform(X_test)

# Save TF-IDF Vectorizer for Email Module
joblib.dump(tfidf_email, '../models/email/tfidf_vectorizer.pkl')
print("✅ Saved Email TF-IDF Vectorizer to 'models/email/tfidf_vectorizer.pkl'")

# 4. Train Classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=500, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=20, class_weight='balanced', random_state=42, n_jobs=-1)
}

results = []

for name, clf in classifiers.items():
    print(f"\nTraining {name}...")
    clf.fit(X_train_tfidf, y_train)
    
    y_pred = clf.predict(X_test_tfidf)
    y_proba = clf.predict_proba(X_test_tfidf)[:, 1] if hasattr(clf, "predict_proba") else y_pred
    
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)
    
    results.append({
        "Model": name,
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc, 4)
    })
    
    # Save Model
    filename = f"../models/email/{name.lower().replace(' ', '_')}.pkl"
    joblib.dump(clf, filename)

# Print Summary Table
results_df = pd.DataFrame(results)
print("\n" + "="*50)
print("     MODULE 2: EMAIL MODEL COMPARISON REPORT")
print("="*50)
print(results_df.to_string(index=False))

Legitimate Emails (0): 17312 | Phishing Emails (1): 21842

Extracting features using TF-IDF...
✅ Saved Email TF-IDF Vectorizer to 'models/email/tfidf_vectorizer.pkl'

Training Logistic Regression...

Training Multinomial Naive Bayes...

Training Random Forest...

     MODULE 2: EMAIL MODEL COMPARISON REPORT
                  Model  Precision  Recall  F1-Score  ROC-AUC
    Logistic Regression     0.9957  0.9963    0.9960   0.9997
Multinomial Naive Bayes     0.9969  0.9423    0.9688   0.9987
          Random Forest     0.9732  0.9963    0.9846   0.9989


In [4]:
# Load trained model and vectorizer
lr_email = joblib.load('../models/email/logistic_regression.pkl')
vectorizer_email = joblib.load('../models/email/tfidf_vectorizer.pkl')

feature_names = np.array(vectorizer_email.get_feature_names_out())
coefficients = lr_email.coef_[0]

def predict_and_explain_email(email_text):
    text_tfidf = vectorizer_email.transform([email_text])
    pred = lr_email.predict(text_tfidf)[0]
    prob = lr_email.predict_proba(text_tfidf)[0][1]
    
    non_zero_indices = text_tfidf.nonzero()[1]
    word_weights = []
    
    for idx in non_zero_indices:
        word = feature_names[idx]
        weight = coefficients[idx]
        word_weights.append((word, weight))
    
    suspicious_indicators = sorted([item for item in word_weights if item[1] > 0], key=lambda x: x[1], reverse=True)[:5]
    
    print("\n" + "="*50)
    print("   MODULE 2: EXPLAINABLE EMAIL FRAUD RESULT")
    print("="*50)
    print(f"Status           : {'🚨 PHISHING / FAKE INTERVIEW EMAIL' if pred == 1 else '✅ LEGITIMATE EMAIL'}")
    print(f"Fraud Risk Score : {prob * 100:.2f}%")
    
    if suspicious_indicators:
        print("\nTop Suspicious Risk Triggers:")
        for word, score in suspicious_indicators:
            print(f" - '{word}' (Risk Weight: +{score:.3f})")
    else:
        print("\nNo high-risk email fraud triggers detected.")
    print("="*50)

# Test with fake interview email prompt
sample_fake_email = """
Urgent candidate interview selection: Your application has been approved. 
Click here to claim your job offer and transfer processing fees to finalize your slot.
"""

predict_and_explain_email(sample_fake_email)


   MODULE 2: EXPLAINABLE EMAIL FRAUD RESULT
Status           : 🚨 PHISHING / FAKE INTERVIEW EMAIL
Fraud Risk Score : 90.75%

Top Suspicious Risk Triggers:
 - 'offer' (Risk Weight: +2.293)
 - 'approved' (Risk Weight: +1.003)
 - 'selection' (Risk Weight: +0.619)
 - 'click' (Risk Weight: +0.554)
 - 'processing' (Risk Weight: +0.331)
